In [ ]:
import sqlite3
import pandas as pd
import sys
import os

# Thêm đường dẫn để import được file từ thư mục dashboard
sys.path.append(os.path.abspath('../dashboard'))
from sleep_rules import calculate_stress_level, calculate_movement_score

# Đọc toàn bộ dữ liệu thật từ SQLite
conn = sqlite3.connect('../data/database/sleep_records.db')
df_raw = pd.read_sql_query("SELECT * FROM sensor_data", conn)
conn.close()

# Chuyển cột timestamp sang định dạng datetime chuẩn
df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])

# Chuyển timestamp làm index để dễ dàng gom nhóm theo thời gian (Resample)
df_raw.set_index('timestamp', inplace=True)

# Gom nhóm dữ liệu theo từng 1 phút ('1min')
dataset_rows = []

for time_window, group in df_raw.resample('1min'):
    if len(group) < 5: 
        continue # Bỏ qua các phút bị rớt mạng, có quá ít dữ liệu
        
    # Tính các thông số trung bình trong 1 phút đó
    avg_hr = group['heart_rate'].mean()
    avg_spo2 = group['spo2'].mean()
    avg_lux = group['light_lux'].mean()
    
    # 1. Gọi hàm tính Stress từ thư viện của bạn
    stress_score = calculate_stress_level(group['heart_rate'].values, group['spo2'].values)
    
    # 2. Gọi hàm tính Điểm vận động từ thư viện của bạn
    mov_score, mov_status = calculate_movement_score(group[['accel_x', 'accel_y', 'accel_z']])
    
    # Đóng gói thành 1 dòng dữ liệu hoàn chỉnh cho AI
    dataset_rows.append({
        'Timestamp': time_window,
        'Heart_Rate_Avg': round(avg_hr, 1),
        'SpO2_Avg': round(avg_spo2, 1),
        'Light_Lux_Avg': round(avg_lux, 1),
        'Stress_Level': stress_score,
        'Movement_Score': mov_score,
        'Movement_Status': mov_status
    })

# Xuất ra file CSV Dataset mới
df_final_dataset = pd.DataFrame(dataset_rows)
df_final_dataset.to_csv('../data/processed/my_real_sleep_dataset.csv', index=False)

print("Đã tạo thành công Dataset thực tế!")
print(f"Tổng số mẫu thu được: {len(df_final_dataset)} phút giấc ngủ.")
df_final_dataset.head()